# 5. Data Warehouse – Athena / Glue Data Catalog

En la pauta original, esta etapa corresponde a guardar el resultado procesado en BigQuery y construir una capa analítica.

En esta implementación equivalente en AWS, el resultado procesado generado en la etapa anterior fue almacenado en Amazon S3 en formato Parquet, dentro de la zona `processed/`.

A partir de esa salida procesada, se construirá una capa analítica utilizando Amazon Athena y AWS Glue Data Catalog. Esto permitirá consultar el dataset procesado mediante SQL, de forma similar a una tabla analítica en BigQuery.

Flujo equivalente:

```text
Amazon S3 processed
        ↓
Athena / Glue Data Catalog
        ↓
Tabla analítica procesada
        ↓
Consultas SQL de indicadores
```

Este notebook continúa desde la salida generada en `pipeline_batch_salmonicultura02.ipynb`.

## 5.1 Reconexión al entorno AWS

Como el laboratorio AWS utiliza credenciales temporales, al iniciar este notebook es necesario volver a cargar la configuración local.

No se recrearán los recursos ya generados en etapas anteriores.  
Solo se cargarán nuevamente las credenciales y rutas desde el archivo `.env`, y se validará que los recursos en AWS sigan disponibles.

Acciones de esta sección:

1. Cargar variables del archivo `.env`.
2. Crear clientes AWS para S3, Athena y STS.
3. Validar credenciales temporales.
4. Comprobar que existe la carpeta procesada en S3.
5. Continuar con la creación de la capa analítica.

In [1]:
# Esta celda carga las credenciales temporales del laboratorio AWS y las rutas del proyecto.
# No crea recursos en AWS; solo prepara las variables locales necesarias para continuar.

from dotenv import load_dotenv
import os
import boto3
import time

load_dotenv(override=True)

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN")
AWS_DEFAULT_REGION = os.getenv("AWS_DEFAULT_REGION")

S3_BUCKET = os.getenv("S3_BUCKET")
S3_RAW_PREFIX = os.getenv("S3_RAW_PREFIX")
S3_RAW_KEY = os.getenv("S3_RAW_KEY")
S3_PROCESSED_PREFIX = os.getenv("S3_PROCESSED_PREFIX", "processed/productive_kpi/")
S3_EXPORTS_PREFIX = os.getenv("S3_EXPORTS_PREFIX", "exports/productive_kpi_csv/")

ATHENA_DATABASE = os.getenv("ATHENA_DATABASE")
ATHENA_RAW_TABLE = os.getenv("ATHENA_RAW_TABLE")
ATHENA_OUTPUT = os.getenv("ATHENA_OUTPUT")

ATHENA_DW_DATABASE = os.getenv("ATHENA_DW_DATABASE", "salmonicultura_dw_db")
ATHENA_PROCESSED_TABLE = os.getenv("ATHENA_PROCESSED_TABLE", "productive_kpi_processed")

S3_PROCESSED_PATH = f"s3://{S3_BUCKET}/{S3_PROCESSED_PREFIX}"

print("Región AWS:", AWS_DEFAULT_REGION)
print("Bucket:", S3_BUCKET)
print("Ruta processed:", S3_PROCESSED_PATH)
print("Base DW Athena:", ATHENA_DW_DATABASE)
print("Tabla procesada:", ATHENA_PROCESSED_TABLE)
print("Salida Athena:", ATHENA_OUTPUT)

Región AWS: us-east-1
Bucket: bigdata-salmonicultura-fabian
Ruta processed: s3://bigdata-salmonicultura-fabian/processed/productive_kpi/
Base DW Athena: salmonicultura_dw_db
Tabla procesada: productive_kpi_processed
Salida Athena: s3://bigdata-salmonicultura-fabian/exports/athena_results/


In [2]:
# Esta celda crea los clientes AWS necesarios para este notebook.
# STS valida credenciales, S3 revisa archivos y Athena ejecuta consultas SQL.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
    region_name=AWS_DEFAULT_REGION
)

athena = boto3.client(
    "athena",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
    region_name=AWS_DEFAULT_REGION
)

sts = boto3.client(
    "sts",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
    region_name=AWS_DEFAULT_REGION
)

print("Clientes AWS creados correctamente.")

Clientes AWS creados correctamente.


In [3]:
# Esta celda valida que las credenciales temporales del laboratorio AWS estén activas.

identity = sts.get_caller_identity()

print("Credenciales AWS válidas.")
print("Cuenta AWS:", identity["Account"])
print("ARN:", identity["Arn"])

Credenciales AWS válidas.
Cuenta AWS: 601270941236
ARN: arn:aws:sts::601270941236:assumed-role/voclabs/user3277614=fabi.lecaros@duocuc.cl


In [4]:
# Esta celda comprueba que la salida procesada generada en el notebook anterior existe en S3.
# Deben aparecer archivos Parquet dentro de la carpeta processed/productive_kpi/.

response_processed = s3.list_objects_v2(
    Bucket=S3_BUCKET,
    Prefix=S3_PROCESSED_PREFIX
)

print("Archivos encontrados en la zona processed:")

if "Contents" in response_processed:
    for obj in response_processed["Contents"]:
        print("-", obj["Key"], "| Tamaño:", obj["Size"], "bytes")
else:
    print("No se encontraron archivos en la zona processed.")

Archivos encontrados en la zona processed:
- processed/productive_kpi/_SUCCESS | Tamaño: 0 bytes
- processed/productive_kpi/part-00000-c56b4d23-f292-4f33-9196-7ead4247c85e-c000.snappy.parquet | Tamaño: 4137735 bytes
- processed/productive_kpi/part-00001-c56b4d23-f292-4f33-9196-7ead4247c85e-c000.snappy.parquet | Tamaño: 4156860 bytes
- processed/productive_kpi/part-00002-c56b4d23-f292-4f33-9196-7ead4247c85e-c000.snappy.parquet | Tamaño: 2742962 bytes
- processed/productive_kpi/part-00003-c56b4d23-f292-4f33-9196-7ead4247c85e-c000.snappy.parquet | Tamaño: 3911160 bytes
- processed/productive_kpi/part-00004-c56b4d23-f292-4f33-9196-7ead4247c85e-c000.snappy.parquet | Tamaño: 4003647 bytes
- processed/productive_kpi/part-00005-c56b4d23-f292-4f33-9196-7ead4247c85e-c000.snappy.parquet | Tamaño: 3935212 bytes
- processed/productive_kpi/part-00006-c56b4d23-f292-4f33-9196-7ead4247c85e-c000.snappy.parquet | Tamaño: 3946851 bytes
- processed/productive_kpi/part-00007-c56b4d23-f292-4f33-9196-7ead4247

### Resultado de la reconexión

La conexión al laboratorio AWS fue restablecida correctamente.

Se validaron las credenciales temporales, se crearon los clientes necesarios para trabajar con S3 y Athena, y se comprobó que la salida procesada en formato Parquet sigue disponible en la zona `processed/` del bucket S3.

Con esto, el notebook queda preparado para construir la capa analítica sobre el dataset procesado.

## 5.2 Creación de base de datos analítica en Athena

En esta sección se crea una base de datos analítica en Athena.  
Esta base de datos cumple el rol equivalente al Data Warehouse en BigQuery dentro de la pauta original.

La base analítica almacenará la definición de la tabla procesada, la cual apuntará directamente a los archivos Parquet ubicados en la zona `processed/` de Amazon S3.

In [5]:
# Esta función ejecuta consultas SQL en Athena desde el notebook.
# Athena ejecuta consultas de forma asíncrona, por lo que la función espera hasta que la consulta termine.

def run_athena_query(query, database=None):
    response = athena.start_query_execution(
        QueryString=query,
        QueryExecutionContext={
            "Database": database or ATHENA_DW_DATABASE
        },
        ResultConfiguration={
            "OutputLocation": ATHENA_OUTPUT
        }
    )

    query_execution_id = response["QueryExecutionId"]

    while True:
        result = athena.get_query_execution(
            QueryExecutionId=query_execution_id
        )

        state = result["QueryExecution"]["Status"]["State"]

        if state in ["SUCCEEDED", "FAILED", "CANCELLED"]:
            break

        time.sleep(2)

    if state != "SUCCEEDED":
        reason = result["QueryExecution"]["Status"].get("StateChangeReason", "Sin detalle")
        raise Exception(f"La consulta falló. Estado: {state}. Motivo: {reason}")

    return query_execution_id

In [6]:
# Esta celda crea la base de datos analítica en Athena si todavía no existe.
# No carga datos físicamente; solo crea el contenedor lógico donde quedará registrada la tabla procesada.

create_dw_database_query = f"""
CREATE DATABASE IF NOT EXISTS {ATHENA_DW_DATABASE}
"""

query_id = run_athena_query(create_dw_database_query)

print("Base de datos analítica creada o ya existente.")
print("Base de datos:", ATHENA_DW_DATABASE)
print("QueryExecutionId:", query_id)

Base de datos analítica creada o ya existente.
Base de datos: salmonicultura_dw_db
QueryExecutionId: d3aa2ffe-b5d9-41e8-b289-68ad35579400


## 5.3 Creación de tabla analítica procesada

A continuación, se crea una tabla externa en Athena sobre los archivos Parquet generados en la etapa de transformación.

Esta tabla representa la capa analítica del pipeline.  
A diferencia de la tabla RAW, esta tabla contiene datos limpios, tipados y enriquecidos con indicadores productivos orientados al análisis de alimentación.

La tabla no copia los datos desde S3. Athena registra la estructura de la tabla y consulta directamente los archivos Parquet almacenados en la zona `processed/`.

In [7]:
# Esta celda elimina la definición anterior de la tabla procesada si existe.
# No elimina los archivos Parquet en S3; solo remueve la definición de la tabla en Athena.
# Esto permite recrear la tabla con un esquema actualizado si hubo cambios en la transformación.

drop_processed_table_query = f"""
DROP TABLE IF EXISTS {ATHENA_DW_DATABASE}.{ATHENA_PROCESSED_TABLE}
"""

query_id = run_athena_query(drop_processed_table_query)

print("Tabla procesada anterior eliminada si existía.")
print("QueryExecutionId:", query_id)

Tabla procesada anterior eliminada si existía.
QueryExecutionId: b3c0dfd6-0c98-4472-bd20-0fcdf384321a


In [8]:
# Esta celda crea una tabla externa en Athena apuntando a los archivos Parquet procesados en S3.
# El esquema corresponde al dataset final generado por PySpark en la etapa 4.

processed_s3_location = f"s3://{S3_BUCKET}/{S3_PROCESSED_PREFIX}"

create_processed_table_query = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {ATHENA_DW_DATABASE}.{ATHENA_PROCESSED_TABLE} (
    site string,
    unit string,
    year_value bigint,
    week bigint,
    production_week_id string,
    species string,
    year_class string,
    open_count bigint,
    open_biomass double,
    open_weight double,
    feed_weight_numeric double,
    temperature_avg double,
    temperature_level string,
    density_avg double,
    density_level string,
    live_days bigint,
    fish_days bigint,
    mortality_count bigint,
    mortality_rate double,
    harvest_count bigint,
    ship_out_count string,
    ship_in_count string,
    close_biomass double,
    biomass_variation double,
    biomass_growth_rate double,
    feed_per_open_biomass double,
    feed_per_fish double,
    data_quality_flag string
)
STORED AS PARQUET
LOCATION '{processed_s3_location}'
"""

query_id = run_athena_query(create_processed_table_query)

print("Tabla analítica procesada creada correctamente.")
print("Tabla:", f"{ATHENA_DW_DATABASE}.{ATHENA_PROCESSED_TABLE}")
print("Ubicación S3:", processed_s3_location)
print("QueryExecutionId:", query_id)

Tabla analítica procesada creada correctamente.
Tabla: salmonicultura_dw_db.productive_kpi_processed
Ubicación S3: s3://bigdata-salmonicultura-fabian/processed/productive_kpi/
QueryExecutionId: 16deb23d-9fe8-4476-99e3-9ba312e162bf


## 5.4 Validación de la tabla analítica

Después de crear la tabla externa, se ejecutan consultas de validación para comprobar que Athena puede leer correctamente los archivos Parquet procesados.

Las validaciones incluyen:

1. Listar las tablas disponibles en la base analítica.
2. Consultar las primeras filas de la tabla procesada.
3. Contar la cantidad total de registros.
4. Confirmar que los indicadores generados están disponibles para análisis.

In [9]:
# Esta celda lista las tablas registradas en la base de datos analítica.
# Sirve para comprobar que la tabla procesada fue registrada correctamente en Athena.

show_tables_query = f"""
SHOW TABLES IN {ATHENA_DW_DATABASE}
"""

query_id = run_athena_query(show_tables_query)

results = athena.get_query_results(QueryExecutionId=query_id)

print("Tablas encontradas en la base analítica:")

for row in results["ResultSet"]["Rows"]:
    values = [col.get("VarCharValue", "") for col in row["Data"]]
    print(values)

Tablas encontradas en la base analítica:
['productive_kpi_processed']


In [10]:
# Esta celda consulta las primeras filas de la tabla analítica procesada.
# Si la consulta devuelve registros, la capa analítica está funcionando correctamente.

select_processed_query = f"""
SELECT *
FROM {ATHENA_DW_DATABASE}.{ATHENA_PROCESSED_TABLE}
LIMIT 10
"""

query_id = run_athena_query(select_processed_query)

results = athena.get_query_results(QueryExecutionId=query_id)

for row in results["ResultSet"]["Rows"]:
    values = [col.get("VarCharValue", "") for col in row["Data"]]
    print(values)

['site', 'unit', 'year_value', 'week', 'production_week_id', 'species', 'year_class', 'open_count', 'open_biomass', 'open_weight', 'feed_weight_numeric', 'temperature_avg', 'temperature_level', 'density_avg', 'density_level', 'live_days', 'fish_days', 'mortality_count', 'mortality_rate', 'harvest_count', 'ship_out_count', 'ship_in_count', 'close_biomass', 'biomass_variation', 'biomass_growth_rate', 'feed_per_open_biomass', 'feed_per_fish', 'data_quality_flag']
['Site_088', 'Unit_3540', '2012', '', '2012', 'Salar', '2012-ANUAL', '317', '0.313380003', '0.988580451', '0.105999999', '10.54285703', 'media', '2.047087694', 'baja', '7', '2219', '', '', '', '0', '0', '0.423796684', '0.11041668099999996', '0.35234118304606676', '0.3382474886248565', '3.343848548895899E-4', 'invalid_key']
['Site_088', 'Unit_3842', '2008', '', '2008', 'Salar', '2007-ANUAL', '555', '9.313908577', '16.78181726', '1.08', '11.84285709', 'media', '0.0', 'baja', '7', '3859', '5', '0.009009009009009009', '', '0', '0', '

In [11]:
# Esta celda cuenta los registros disponibles en la tabla procesada.
# El resultado permite validar que Athena está leyendo la salida Parquet generada por PySpark.

count_processed_query = f"""
SELECT COUNT(*) AS total_registros
FROM {ATHENA_DW_DATABASE}.{ATHENA_PROCESSED_TABLE}
"""

query_id = run_athena_query(count_processed_query)

results = athena.get_query_results(QueryExecutionId=query_id)

for row in results["ResultSet"]["Rows"]:
    values = [col.get("VarCharValue", "") for col in row["Data"]]
    print(values)

['total_registros']
['783244']


## 5.5 Consultas analíticas sobre alimentación y producción

Con la tabla procesada ya disponible en Athena, se ejecutan consultas SQL para obtener indicadores de análisis productivo.

Estas consultas permiten utilizar la capa analítica para responder preguntas relacionadas con alimentación, biomasa, mortalidad, temperatura y densidad.

In [14]:
# Esta consulta calcula el alimento total entregado por centro.
# Se convierte el resultado a DECIMAL para evitar que Athena muestre números grandes en notación científica.

query_total_feed_by_site = f"""
SELECT
    site,
    CAST(ROUND(SUM(feed_weight_numeric), 2) AS DECIMAL(18,2)) AS total_feed_weight
FROM {ATHENA_DW_DATABASE}.{ATHENA_PROCESSED_TABLE}
WHERE feed_weight_numeric IS NOT NULL
GROUP BY site
ORDER BY total_feed_weight DESC
"""

query_id = run_athena_query(query_total_feed_by_site)

results = athena.get_query_results(QueryExecutionId=query_id)

for row in results["ResultSet"]["Rows"]:
    values = [col.get("VarCharValue", "") for col in row["Data"]]
    print(values)

['site', 'total_feed_weight']
['Site_038', '92833689.00']
['Site_059', '85706240.32']
['Site_012', '62291237.00']
['Site_027', '50881250.61']
['Site_049', '48557404.59']
['Site_062', '47776374.00']
['Site_039', '45869924.00']
['Site_029', '36624624.95']
['Site_048', '35300805.50']
['Site_033', '34584326.00']
['Site_055', '33637967.00']
['Site_064', '31287314.00']
['Site_047', '29318397.00']
['Site_041', '27431481.00']
['Site_025', '26173905.00']
['Site_017', '25381920.00']
['Site_053', '25300302.00']
['Site_046', '24570398.00']
['Site_043', '22962778.98']
['Site_036', '19309254.00']
['Site_101', '18355514.80']
['Site_015', '17128063.00']
['Site_050', '15170791.00']
['Site_045', '14946936.00']
['Site_034', '14851572.00']
['Site_040', '14528925.00']
['Site_066', '14416513.00']
['Site_018', '14347618.00']
['Site_060', '14099096.00']
['Site_073', '12404372.00']
['Site_098', '12135802.58']
['Site_056', '11337759.00']
['Site_021', '11099769.00']
['Site_032', '10410205.00']
['Site_082', '1008

In [15]:
# Esta consulta calcula el promedio de alimento entregado por biomasa inicial en cada centro.
# Solo considera registros donde existe biomasa inicial válida para evitar divisiones inválidas.

query_feed_per_biomass_by_site = f"""
SELECT
    site,
    COUNT(*) AS registros_validos,
    CAST(ROUND(AVG(feed_per_open_biomass), 6) AS DECIMAL(18,6)) AS avg_feed_per_open_biomass,
    CAST(ROUND(SUM(feed_weight_numeric), 2) AS DECIMAL(18,2)) AS total_feed_weight,
    CAST(ROUND(SUM(open_biomass), 2) AS DECIMAL(18,2)) AS total_open_biomass
FROM {ATHENA_DW_DATABASE}.{ATHENA_PROCESSED_TABLE}
WHERE feed_per_open_biomass IS NOT NULL
  AND open_biomass > 0
GROUP BY site
ORDER BY avg_feed_per_open_biomass DESC
"""

query_id = run_athena_query(query_feed_per_biomass_by_site)

results = athena.get_query_results(QueryExecutionId=query_id)

for row in results["ResultSet"]["Rows"]:
    values = [col.get("VarCharValue", "") for col in row["Data"]]
    print(values)

['site', 'registros_validos', 'avg_feed_per_open_biomass', 'total_feed_weight', 'total_open_biomass']
['Site_098', '13438', '12273157470.485428', '11558784.63', '90675341.46']
['Site_088', '186776', '423848482.873425', '1001302.27', '29111004.43']
['Site_007', '17612', '994.255325', '1096904.44', '7736360.39']
['Site_010', '8', '2.525223', '195347.00', '367120.72']
['Site_006', '5', '2.509064', '10234.70', '4390.55']
['Site_050', '1975', '1.516254', '15105713.00', '237398841.55']
['Site_004', '741', '0.362136', '18091.88', '76110.04']
['Site_005', '116883', '0.295311', '6638356.36', '45786462.11']
['Site_074', '11', '0.281476', '289511.00', '1461244.15']
['Site_090', '264', '0.223832', '15553.24', '231420.12']
['Site_084', '4686', '0.165548', '85210.31', '563659.72']
['Site_001', '845', '0.160071', '354258.00', '3240459.47']
['Site_100', '259', '0.146564', '163980.00', '1402609.46']
['Site_003', '644', '0.131259', '145773.00', '1078402.10']
['Site_083', '1989', '0.110111', '272573.30',

In [16]:
# Esta consulta calcula la variación total de biomasa por centro.
# Permite observar qué centros aumentaron o disminuyeron más biomasa durante el periodo analizado.

query_biomass_variation_by_site = f"""
SELECT
    site,
    COUNT(*) AS registros,
    CAST(ROUND(SUM(biomass_variation), 2) AS DECIMAL(18,2)) AS total_biomass_variation,
    CAST(ROUND(AVG(biomass_growth_rate), 6) AS DECIMAL(18,6)) AS avg_biomass_growth_rate
FROM {ATHENA_DW_DATABASE}.{ATHENA_PROCESSED_TABLE}
WHERE biomass_variation IS NOT NULL
GROUP BY site
ORDER BY total_biomass_variation DESC
"""

query_id = run_athena_query(query_biomass_variation_by_site)

results = athena.get_query_results(QueryExecutionId=query_id)

for row in results["ResultSet"]["Rows"]:
    values = [col.get("VarCharValue", "") for col in row["Data"]]
    print(values)

['site', 'registros', 'total_biomass_variation', 'avg_biomass_growth_rate']
['Site_012', '10943', '4930282.80', '0.035761']
['Site_018', '3312', '4311328.75', '0.047669']
['Site_048', '7693', '4054675.31', '0.035934']
['Site_030', '792', '3321590.79', '0.081726']
['Site_068', '461', '3167870.14', '0.049631']
['Site_050', '2287', '2598306.16', '1.041523']
['Site_027', '10508', '2422322.41', '0.033945']
['Site_040', '2843', '2018023.60', '0.048775']
['Site_049', '10142', '1574486.23', '0.031915']
['Site_041', '6564', '1409864.86', '0.044753']
['Site_025', '6560', '798326.21', '0.035264']
['Site_101', '32340', '543858.32', '0.080898']
['Site_046', '5340', '465761.15', '0.040849']
['Site_036', '3917', '300018.18', '0.029349']
['Site_005', '212822', '63239.55', '584.454684']
['Site_038', '17836', '31004.36', '0.034213']
['Site_088', '207167', '29942.48', '12231524994.852110']
['Site_002', '446', '28030.55', '0.084426']
['Site_091', '14560', '27386.84', '379.073355']
['Site_006', '583', '106

In [17]:
# Esta consulta calcula la mortalidad promedio por centro.
# Solo considera registros donde la tasa de mortalidad pudo calcularse correctamente.

query_mortality_by_site = f"""
SELECT
    site,
    COUNT(*) AS registros_validos,
    CAST(ROUND(AVG(mortality_rate) * 100, 4) AS DECIMAL(18,4)) AS avg_mortality_percent,
    CAST(ROUND(SUM(mortality_count), 2) AS DECIMAL(18,2)) AS total_mortality_count
FROM {ATHENA_DW_DATABASE}.{ATHENA_PROCESSED_TABLE}
WHERE mortality_rate IS NOT NULL
GROUP BY site
ORDER BY avg_mortality_percent DESC
"""

query_id = run_athena_query(query_mortality_by_site)

results = athena.get_query_results(QueryExecutionId=query_id)

for row in results["ResultSet"]["Rows"]:
    values = [col.get("VarCharValue", "") for col in row["Data"]]
    print(values)

['site', 'registros_validos', 'avg_mortality_percent', 'total_mortality_count']
['Site_014', '1', '100.0000', '3526.00']
['Site_023', '1', '100.0000', '52227.00']
['Site_091', '5686', '17.8483', '169873227.00']
['Site_092', '1461', '15.5482', '20271478.00']
['Site_071', '10', '10.2291', '19634.00']
['Site_099', '10', '10.1932', '94780.00']
['Site_088', '56526', '9.4360', '1865433.00']
['Site_052', '24', '8.6782', '67270.00']
['Site_058', '14', '7.6724', '42312.00']
['Site_090', '176', '7.3074', '11688.00']
['Site_100', '291', '4.0893', '1161350.00']
['Site_070', '30', '3.6063', '20951.00']
['Site_004', '1124', '3.3559', '1571649.00']
['Site_077', '334', '2.8715', '479417.00']
['Site_001', '903', '2.7902', '954767.00']
['Site_084', '6773', '2.4452', '5424705.00']
['Site_074', '16', '2.2506', '24515.00']
['Site_083', '2130', '2.1630', '1141511.00']
['Site_087', '1281', '1.9131', '122929.00']
['Site_008', '5790', '1.7571', '4989730.00']
['Site_089', '370', '1.6065', '52633.00']
['Site_096

In [18]:
# Esta consulta analiza el alimento relativo a biomasa según nivel de temperatura.
# Permite observar si existen diferencias generales en alimentación y crecimiento bajo distintos rangos de temperatura.

query_feed_by_temperature_level = f"""
SELECT
    temperature_level,
    COUNT(*) AS registros,
    CAST(ROUND(AVG(feed_per_open_biomass), 6) AS DECIMAL(18,6)) AS avg_feed_per_open_biomass,
    CAST(ROUND(AVG(biomass_growth_rate), 6) AS DECIMAL(18,6)) AS avg_biomass_growth_rate,
    CAST(ROUND(SUM(feed_weight_numeric), 2) AS DECIMAL(18,2)) AS total_feed_weight
FROM {ATHENA_DW_DATABASE}.{ATHENA_PROCESSED_TABLE}
WHERE temperature_level IS NOT NULL
  AND feed_per_open_biomass IS NOT NULL
GROUP BY temperature_level
ORDER BY temperature_level
"""

query_id = run_athena_query(query_feed_by_temperature_level)

results = athena.get_query_results(QueryExecutionId=query_id)

for row in results["ResultSet"]["Rows"]:
    values = [col.get("VarCharValue", "") for col in row["Data"]]
    print(values)

['temperature_level', 'registros', 'avg_feed_per_open_biomass', 'avg_biomass_growth_rate', 'total_feed_weight']
['alta', '122634', '0.235468', '0.199869', '102962458.91']
['baja', '118681', '667037893.338962', '20436196123.715508', '170895352.22']
['media', '358877', '459563325.721665', '13497803061.235998', '848018614.93']
['sin_dato', '95', '0.025714', '-0.998377', '6779.19']


In [19]:
# Esta consulta analiza el alimento relativo a biomasa según nivel de densidad.
# Permite observar si las unidades con mayor densidad presentan diferencias en alimentación y crecimiento.

query_feed_by_density_level = f"""
SELECT
    density_level,
    COUNT(*) AS registros,
    CAST(ROUND(AVG(feed_per_open_biomass), 6) AS DECIMAL(18,6)) AS avg_feed_per_open_biomass,
    CAST(ROUND(AVG(biomass_growth_rate), 6) AS DECIMAL(18,6)) AS avg_biomass_growth_rate,
    CAST(ROUND(AVG(mortality_rate) * 100, 4) AS DECIMAL(18,4)) AS avg_mortality_percent
FROM {ATHENA_DW_DATABASE}.{ATHENA_PROCESSED_TABLE}
WHERE density_level IS NOT NULL
  AND feed_per_open_biomass IS NOT NULL
GROUP BY density_level
ORDER BY density_level
"""

query_id = run_athena_query(query_feed_by_density_level)

results = athena.get_query_results(QueryExecutionId=query_id)

for row in results["ResultSet"]["Rows"]:
    values = [col.get("VarCharValue", "") for col in row["Data"]]
    print(values)

['density_level', 'registros', 'avg_feed_per_open_biomass', 'avg_biomass_growth_rate', 'avg_mortality_percent']
['alta', '106357', '0.125522', '0.057702', '1.0456']
['baja', '388887', '627666730.613903', '18692934609.214115', '0.8876']
['media', '105043', '0.110920', '0.100612', '1.0739']


In [20]:
# Esta consulta identifica las unidades de cultivo con mayor alimento total entregado.
# Sirve para revisar el comportamiento por centro y unidad, no solo a nivel de sitio.

query_top_units_by_feed = f"""
SELECT
    site,
    unit,
    CAST(ROUND(SUM(feed_weight_numeric), 2) AS DECIMAL(18,2)) AS total_feed_weight,
    CAST(ROUND(AVG(feed_per_open_biomass), 6) AS DECIMAL(18,6)) AS avg_feed_per_open_biomass,
    CAST(ROUND(SUM(biomass_variation), 2) AS DECIMAL(18,2)) AS total_biomass_variation
FROM {ATHENA_DW_DATABASE}.{ATHENA_PROCESSED_TABLE}
WHERE feed_weight_numeric IS NOT NULL
GROUP BY site, unit
ORDER BY total_feed_weight DESC
LIMIT 20
"""

query_id = run_athena_query(query_top_units_by_feed)

results = athena.get_query_results(QueryExecutionId=query_id)

for row in results["ResultSet"]["Rows"]:
    values = [col.get("VarCharValue", "") for col in row["Data"]]
    print(values)

['site', 'unit', 'total_feed_weight', 'avg_feed_per_open_biomass', 'total_biomass_variation']
['Site_038', 'Unit_1659', '3756284.00', '0.078212', '73040.60']
['Site_025', 'Unit_1356', '2135349.00', '0.072668', '-213510.55']
['Site_029', 'Unit_1500', '2124368.40', '0.071094', '1279032.03']
['Site_029', 'Unit_1501', '2083828.00', '0.068397', '1269288.94']
['Site_059', 'Unit_2339', '2079483.00', '0.070844', '1762426.49']
['Site_059', 'Unit_2338', '2026696.00', '0.069682', '1353817.91']
['Site_029', 'Unit_1503', '2010568.20', '0.068202', '1404136.25']
['Site_059', 'Unit_2344', '1995317.00', '0.072869', '1717575.90']
['Site_059', 'Unit_2337', '1984715.00', '0.071310', '1700983.31']
['Site_059', 'Unit_2330', '1982104.00', '0.070389', '1487189.22']
['Site_059', 'Unit_2323', '1971862.00', '0.069991', '1436494.60']
['Site_059', 'Unit_2341', '1967571.10', '0.070338', '1697807.96']
['Site_066', 'Unit_2463', '1966006.00', '0.081139', '1221406.62']
['Site_012', 'Unit_1128', '1959896.00', '0.066080'

## 5.6 Persistencia y Respaldo

Aunque el resultado procesado ya fue guardado en Amazon S3 en formato Parquet, se incluye esta sección para evidenciar explícitamente la persistencia y respaldo solicitados por la pauta.

La capa procesada principal se encuentra almacenada en:

```text
s3://bigdata-salmonicultura-fabian/processed/productive_kpi/
```

Esta carpeta contiene los archivos Parquet generados por PySpark en la etapa de transformación. Estos archivos son utilizados por Athena como base de la tabla analítica procesada.

Además, Athena genera archivos de salida por cada consulta ejecutada, los cuales quedan almacenados en la carpeta configurada como salida de resultados:

```text
s3://bigdata-salmonicultura-fabian/exports/athena_results/
```

De esta forma, el proyecto cuenta con dos formas de persistencia:

| Tipo de persistencia | Ubicación | Uso |
|---|---|---|
| Dataset procesado en Parquet | `processed/productive_kpi/` | Capa analítica principal |
| Resultados de consultas Athena | `exports/athena_results/` | Evidencia y respaldo de consultas SQL |

Con esto se cumple la persistencia del dataset procesado y se deja evidencia de los resultados obtenidos durante la construcción de la capa analítica.

In [21]:
# Esta celda valida la persistencia del dataset procesado y de los resultados de Athena en S3.
# Se listan los archivos almacenados en processed/ y exports/ como evidencia del respaldo.

prefixes_to_check = {
    "Dataset procesado Parquet": S3_PROCESSED_PREFIX,
    "Resultados Athena": "exports/athena_results/"
}

for label, prefix in prefixes_to_check.items():
    print(f"\n{label}")
    print(f"Prefijo S3: s3://{S3_BUCKET}/{prefix}")

    response = s3.list_objects_v2(
        Bucket=S3_BUCKET,
        Prefix=prefix
    )

    if "Contents" in response:
        for obj in response["Contents"][:10]:
            print("-", obj["Key"], "| Tamaño:", obj["Size"], "bytes")
    else:
        print("No se encontraron archivos.")


Dataset procesado Parquet
Prefijo S3: s3://bigdata-salmonicultura-fabian/processed/productive_kpi/
- processed/productive_kpi/_SUCCESS | Tamaño: 0 bytes
- processed/productive_kpi/part-00000-c56b4d23-f292-4f33-9196-7ead4247c85e-c000.snappy.parquet | Tamaño: 4137735 bytes
- processed/productive_kpi/part-00001-c56b4d23-f292-4f33-9196-7ead4247c85e-c000.snappy.parquet | Tamaño: 4156860 bytes
- processed/productive_kpi/part-00002-c56b4d23-f292-4f33-9196-7ead4247c85e-c000.snappy.parquet | Tamaño: 2742962 bytes
- processed/productive_kpi/part-00003-c56b4d23-f292-4f33-9196-7ead4247c85e-c000.snappy.parquet | Tamaño: 3911160 bytes
- processed/productive_kpi/part-00004-c56b4d23-f292-4f33-9196-7ead4247c85e-c000.snappy.parquet | Tamaño: 4003647 bytes
- processed/productive_kpi/part-00005-c56b4d23-f292-4f33-9196-7ead4247c85e-c000.snappy.parquet | Tamaño: 3935212 bytes
- processed/productive_kpi/part-00006-c56b4d23-f292-4f33-9196-7ead4247c85e-c000.snappy.parquet | Tamaño: 3946851 bytes
- processed/p

# 6. Analítica Predictiva – Preparación en Athena y modelo local

La pauta original propone BigQuery ML como etapa opcional. BigQuery ML permite crear modelos predictivos directamente desde el entorno analítico mediante SQL.

En esta implementación AWS, el equivalente más cercano sería Amazon Redshift ML. Sin embargo, debido a las limitaciones del laboratorio y para mantener el flujo desarrollado con S3, Athena y PySpark, se implementa una aproximación mixta:

1. Athena prepara y persiste los datasets de entrenamiento y prueba.
2. El modelo se entrena localmente desde el notebook.
3. El modelo, las métricas y las predicciones se almacenan nuevamente en Amazon S3.
4. Las predicciones se registran como tabla Athena para poder ser usadas posteriormente en dashboards.

El objetivo del modelo será clasificar semanas productivas con bajo crecimiento de biomasa, utilizando variables relacionadas con alimentación, temperatura, densidad, mortalidad y biomasa inicial.

## 6.1 Variables y rutas para analítica predictiva

Para esta etapa se utilizarán carpetas adicionales en Amazon S3:

| Carpeta | Uso |
|---|---|
| `ml/dataset/` | Dataset base para machine learning |
| `ml/train/` | Dataset de entrenamiento |
| `ml/test/` | Dataset de prueba |
| `models/` | Modelo entrenado y métricas |
| `exports/ml_predictions/` | Predicciones exportadas para consulta o dashboard |

Estas salidas permiten mantener la evidencia del proceso predictivo dentro de AWS.

In [22]:
# Esta celda define las rutas S3 que se usarán para la etapa de machine learning.
# Se agregan rutas para dataset ML, entrenamiento, prueba, modelo y predicciones.

S3_ML_DATASET_PREFIX = os.getenv("S3_ML_DATASET_PREFIX", "ml/dataset/")
S3_ML_TRAIN_PREFIX = os.getenv("S3_ML_TRAIN_PREFIX", "ml/train/")
S3_ML_TEST_PREFIX = os.getenv("S3_ML_TEST_PREFIX", "ml/test/")
S3_MODELS_PREFIX = os.getenv("S3_MODELS_PREFIX", "models/")
S3_ML_PREDICTIONS_PREFIX = os.getenv("S3_ML_PREDICTIONS_PREFIX", "exports/ml_predictions/")

ATHENA_ML_DATASET_TABLE = os.getenv("ATHENA_ML_DATASET_TABLE", "productive_ml_dataset")
ATHENA_ML_TRAIN_TABLE = os.getenv("ATHENA_ML_TRAIN_TABLE", "productive_ml_train")
ATHENA_ML_TEST_TABLE = os.getenv("ATHENA_ML_TEST_TABLE", "productive_ml_test")
ATHENA_ML_PREDICTIONS_TABLE = os.getenv("ATHENA_ML_PREDICTIONS_TABLE", "productive_ml_predictions")

print("Dataset ML:", f"s3://{S3_BUCKET}/{S3_ML_DATASET_PREFIX}")
print("Train ML:", f"s3://{S3_BUCKET}/{S3_ML_TRAIN_PREFIX}")
print("Test ML:", f"s3://{S3_BUCKET}/{S3_ML_TEST_PREFIX}")
print("Modelos:", f"s3://{S3_BUCKET}/{S3_MODELS_PREFIX}")
print("Predicciones:", f"s3://{S3_BUCKET}/{S3_ML_PREDICTIONS_PREFIX}")

Dataset ML: s3://bigdata-salmonicultura-fabian/ml/dataset/
Train ML: s3://bigdata-salmonicultura-fabian/ml/train/
Test ML: s3://bigdata-salmonicultura-fabian/ml/test/
Modelos: s3://bigdata-salmonicultura-fabian/models/
Predicciones: s3://bigdata-salmonicultura-fabian/exports/ml_predictions/


In [23]:
# Esta función elimina archivos dentro de un prefijo S3.
# Se usa antes de crear tablas CTAS en Athena, porque Athena requiere que la carpeta destino esté vacía.

def delete_s3_prefix(bucket, prefix):
    response = s3.list_objects_v2(
        Bucket=bucket,
        Prefix=prefix
    )

    if "Contents" not in response:
        print(f"No había archivos para eliminar en s3://{bucket}/{prefix}")
        return

    objects_to_delete = [
        {"Key": obj["Key"]}
        for obj in response["Contents"]
    ]

    s3.delete_objects(
        Bucket=bucket,
        Delete={"Objects": objects_to_delete}
    )

    print(f"Archivos eliminados en s3://{bucket}/{prefix}")

## 6.2 Creación del dataset base para ML en Athena

Se crea una tabla de machine learning a partir de la tabla analítica procesada.

La variable objetivo será `low_growth_flag`, calculada desde `biomass_growth_rate`.

Se considerará bajo crecimiento cuando `biomass_growth_rate` sea menor que la mediana aproximada del dataset.

Además, se genera una columna `split_rand`, que permitirá separar los datos en entrenamiento y prueba.

In [24]:
# Esta celda elimina la tabla ML previa y limpia su carpeta en S3.
# Esto permite recrear el dataset de ML sin mezclar archivos de ejecuciones anteriores.

drop_ml_dataset_query = f"""
DROP TABLE IF EXISTS {ATHENA_DW_DATABASE}.{ATHENA_ML_DATASET_TABLE}
"""

run_athena_query(drop_ml_dataset_query)

delete_s3_prefix(S3_BUCKET, S3_ML_DATASET_PREFIX)

print("Tabla y carpeta ML base preparadas.")

No había archivos para eliminar en s3://bigdata-salmonicultura-fabian/ml/dataset/
Tabla y carpeta ML base preparadas.


In [25]:
# Esta celda crea una tabla ML base en Athena usando CTAS.
# El dataset queda persistido en S3 en formato Parquet y contiene la variable objetivo low_growth_flag.

ml_dataset_location = f"s3://{S3_BUCKET}/{S3_ML_DATASET_PREFIX}"

create_ml_dataset_query = f"""
CREATE TABLE {ATHENA_DW_DATABASE}.{ATHENA_ML_DATASET_TABLE}
WITH (
    format = 'PARQUET',
    external_location = '{ml_dataset_location}'
) AS
WITH median_value AS (
    SELECT
        approx_percentile(biomass_growth_rate, 0.5) AS growth_median
    FROM {ATHENA_DW_DATABASE}.{ATHENA_PROCESSED_TABLE}
    WHERE feed_per_open_biomass IS NOT NULL
      AND feed_per_fish IS NOT NULL
      AND temperature_avg IS NOT NULL
      AND density_avg IS NOT NULL
      AND mortality_rate IS NOT NULL
      AND open_biomass IS NOT NULL
      AND biomass_growth_rate IS NOT NULL
)
SELECT
    site,
    unit,
    production_week_id,
    feed_per_open_biomass,
    feed_per_fish,
    temperature_avg,
    density_avg,
    mortality_rate,
    open_biomass,
    biomass_growth_rate,
    CASE
        WHEN biomass_growth_rate < median_value.growth_median THEN 1
        ELSE 0
    END AS low_growth_flag,
    rand() AS split_rand
FROM {ATHENA_DW_DATABASE}.{ATHENA_PROCESSED_TABLE}
CROSS JOIN median_value
WHERE feed_per_open_biomass IS NOT NULL
  AND feed_per_fish IS NOT NULL
  AND temperature_avg IS NOT NULL
  AND density_avg IS NOT NULL
  AND mortality_rate IS NOT NULL
  AND open_biomass IS NOT NULL
  AND biomass_growth_rate IS NOT NULL
"""

query_id = run_athena_query(create_ml_dataset_query)

print("Dataset base para ML creado correctamente.")
print("Tabla:", f"{ATHENA_DW_DATABASE}.{ATHENA_ML_DATASET_TABLE}")
print("Ruta S3:", ml_dataset_location)
print("QueryExecutionId:", query_id)

Dataset base para ML creado correctamente.
Tabla: salmonicultura_dw_db.productive_ml_dataset
Ruta S3: s3://bigdata-salmonicultura-fabian/ml/dataset/
QueryExecutionId: 2631a623-e292-441d-9dd0-2acf2e863b56


## 6.3 Separación de entrenamiento y prueba en Athena

A partir del dataset base de ML, se generan dos tablas:

| Tabla | Uso |
|---|---|
| `productive_ml_train` | Entrenamiento del modelo |
| `productive_ml_test` | Evaluación del modelo |

La separación se realiza con la columna `split_rand`, generada y persistida en Athena.

In [26]:
# Esta celda elimina versiones anteriores de las tablas train/test y limpia sus carpetas S3.

for table_name, prefix in [
    (ATHENA_ML_TRAIN_TABLE, S3_ML_TRAIN_PREFIX),
    (ATHENA_ML_TEST_TABLE, S3_ML_TEST_PREFIX)
]:
    drop_query = f"""
    DROP TABLE IF EXISTS {ATHENA_DW_DATABASE}.{table_name}
    """
    run_athena_query(drop_query)
    delete_s3_prefix(S3_BUCKET, prefix)

print("Tablas y carpetas train/test preparadas.")

No había archivos para eliminar en s3://bigdata-salmonicultura-fabian/ml/train/
No había archivos para eliminar en s3://bigdata-salmonicultura-fabian/ml/test/
Tablas y carpetas train/test preparadas.


In [27]:
# Esta celda crea la tabla de entrenamiento en Athena.
# Se usa aproximadamente el 70% de los datos.

train_location = f"s3://{S3_BUCKET}/{S3_ML_TRAIN_PREFIX}"

create_train_query = f"""
CREATE TABLE {ATHENA_DW_DATABASE}.{ATHENA_ML_TRAIN_TABLE}
WITH (
    format = 'PARQUET',
    external_location = '{train_location}'
) AS
SELECT
    site,
    unit,
    production_week_id,
    feed_per_open_biomass,
    feed_per_fish,
    temperature_avg,
    density_avg,
    mortality_rate,
    open_biomass,
    biomass_growth_rate,
    low_growth_flag
FROM {ATHENA_DW_DATABASE}.{ATHENA_ML_DATASET_TABLE}
WHERE split_rand < 0.7
"""

query_id = run_athena_query(create_train_query)

print("Tabla de entrenamiento creada correctamente.")
print("Tabla:", f"{ATHENA_DW_DATABASE}.{ATHENA_ML_TRAIN_TABLE}")
print("Ruta S3:", train_location)
print("QueryExecutionId:", query_id)

Tabla de entrenamiento creada correctamente.
Tabla: salmonicultura_dw_db.productive_ml_train
Ruta S3: s3://bigdata-salmonicultura-fabian/ml/train/
QueryExecutionId: 7447388a-644f-4510-b735-ffd97a0ed12e


In [28]:
# Esta celda crea la tabla de prueba en Athena.
# Se usa aproximadamente el 30% restante de los datos.

test_location = f"s3://{S3_BUCKET}/{S3_ML_TEST_PREFIX}"

create_test_query = f"""
CREATE TABLE {ATHENA_DW_DATABASE}.{ATHENA_ML_TEST_TABLE}
WITH (
    format = 'PARQUET',
    external_location = '{test_location}'
) AS
SELECT
    site,
    unit,
    production_week_id,
    feed_per_open_biomass,
    feed_per_fish,
    temperature_avg,
    density_avg,
    mortality_rate,
    open_biomass,
    biomass_growth_rate,
    low_growth_flag
FROM {ATHENA_DW_DATABASE}.{ATHENA_ML_DATASET_TABLE}
WHERE split_rand >= 0.7
"""

query_id = run_athena_query(create_test_query)

print("Tabla de prueba creada correctamente.")
print("Tabla:", f"{ATHENA_DW_DATABASE}.{ATHENA_ML_TEST_TABLE}")
print("Ruta S3:", test_location)
print("QueryExecutionId:", query_id)

Tabla de prueba creada correctamente.
Tabla: salmonicultura_dw_db.productive_ml_test
Ruta S3: s3://bigdata-salmonicultura-fabian/ml/test/
QueryExecutionId: bc90fe6e-7036-4ad4-b2d2-11ead0e91d6c


In [29]:
# Esta celda valida la cantidad de registros por tabla ML.
# Permite comprobar que existen datos tanto para entrenamiento como para prueba.

for table_name in [
    ATHENA_ML_DATASET_TABLE,
    ATHENA_ML_TRAIN_TABLE,
    ATHENA_ML_TEST_TABLE
]:
    count_query = f"""
    SELECT COUNT(*) AS total_registros
    FROM {ATHENA_DW_DATABASE}.{table_name}
    """

    query_id = run_athena_query(count_query)
    results = athena.get_query_results(QueryExecutionId=query_id)

    print(f"\nTabla: {table_name}")
    for row in results["ResultSet"]["Rows"]:
        values = [col.get("VarCharValue", "") for col in row["Data"]]
        print(values)


Tabla: productive_ml_dataset
['total_registros']
['462432']

Tabla: productive_ml_train
['total_registros']
['323641']

Tabla: productive_ml_test
['total_registros']
['138791']


## 6.4 Entrenamiento local del modelo

El entrenamiento se realiza localmente debido a las limitaciones del laboratorio AWS.

Sin embargo, los datos de entrenamiento y prueba fueron preparados en Athena y persistidos en Amazon S3. Además, el modelo entrenado, sus métricas y las predicciones serán almacenados nuevamente en S3.

El modelo utilizado será una Regresión Logística simple para clasificar semanas con bajo crecimiento de biomasa.

In [30]:
# Esta función ejecuta una consulta Athena y convierte el resultado en un DataFrame de Pandas.
# Se utilizará para traer las tablas de entrenamiento y prueba al notebook local.

import pandas as pd

def athena_query_to_dataframe(query, database=None):
    query_id = run_athena_query(query, database=database)

    rows = []
    next_token = None

    while True:
        if next_token:
            response = athena.get_query_results(
                QueryExecutionId=query_id,
                NextToken=next_token
            )
        else:
            response = athena.get_query_results(
                QueryExecutionId=query_id
            )

        result_rows = response["ResultSet"]["Rows"]

        for row in result_rows:
            values = [col.get("VarCharValue", None) for col in row["Data"]]
            rows.append(values)

        next_token = response.get("NextToken")

        if not next_token:
            break

    columns = rows[0]
    data = rows[1:]

    return pd.DataFrame(data, columns=columns)

In [31]:
# Esta celda carga desde Athena las tablas de entrenamiento y prueba.

train_query = f"""
SELECT *
FROM {ATHENA_DW_DATABASE}.{ATHENA_ML_TRAIN_TABLE}
"""

test_query = f"""
SELECT *
FROM {ATHENA_DW_DATABASE}.{ATHENA_ML_TEST_TABLE}
"""

df_train = athena_query_to_dataframe(train_query)
df_test = athena_query_to_dataframe(test_query)

print("Train:", df_train.shape)
print("Test:", df_test.shape)

df_train.head()

Train: (323641, 11)
Test: (138791, 11)


,site,unit,production_week_id,feed_per_open_biomass,feed_per_fish,temperature_avg,density_avg,mortality_rate,open_biomass,biomass_growth_rate,low_growth_flag
0,Site_101,Unit_4996,2019,0.008115057515472169,0.0020299416391778738,13.8,8.276844808,6.089824917533621E-4,4929.108626,-1.0,1
1,Site_101,Unit_4997,2017,0.005546262512321778,5.102561485865905E-4,13.90714286,16.62978795,0.02839575466884376,3606.031982,1.731223729340734,0
2,Site_101,Unit_4997,2017,0.04467513003796723,0.004092071611253197,13.84285714,33.19099469,0.012955126714717508,9848.880118,0.031605826070630644,1
3,Site_101,Unit_4997,2017,0.08267584620261537,0.007914672294878076,14.17142857,34.60907087,0.0038725360871367733,10160.16211,0.056662020129913164,1
4,Site_101,Unit_4997,2017,0.09566073391462757,0.009714247878850939,14.08571429,37.97349713,0.0010404744563520965,10735.85742,0.11456250878562807,0


In [32]:
# Esta celda convierte a numérico las columnas necesarias para entrenar el modelo.

numeric_columns = [
    "feed_per_open_biomass",
    "feed_per_fish",
    "temperature_avg",
    "density_avg",
    "mortality_rate",
    "open_biomass",
    "biomass_growth_rate",
    "low_growth_flag"
]

for column in numeric_columns:
    df_train[column] = pd.to_numeric(df_train[column], errors="coerce")
    df_test[column] = pd.to_numeric(df_test[column], errors="coerce")

df_train = df_train.dropna(subset=numeric_columns)
df_test = df_test.dropna(subset=numeric_columns)

df_train["low_growth_flag"] = df_train["low_growth_flag"].astype(int)
df_test["low_growth_flag"] = df_test["low_growth_flag"].astype(int)

print("Train limpio:", df_train.shape)
print("Test limpio:", df_test.shape)

Train limpio: (323641, 11)
Test limpio: (138791, 11)


In [33]:
# Esta celda entrena un modelo de Regresión Logística.
# StandardScaler se usa solo dentro del pipeline de ML, no altera la capa analítica almacenada en Athena.

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

feature_columns = [
    "feed_per_open_biomass",
    "feed_per_fish",
    "temperature_avg",
    "density_avg",
    "mortality_rate",
    "open_biomass"
]

target_column = "low_growth_flag"

X_train = df_train[feature_columns]
y_train = df_train[target_column]

X_test = df_test[feature_columns]
y_test = df_test[target_column]

model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

model.fit(X_train, y_train)

print("Modelo entrenado correctamente.")

Modelo entrenado correctamente.


## 6.5 Evaluación del modelo y almacenamiento en S3

Se evalúa el modelo con métricas simples y se almacenan los resultados en Amazon S3.

Esto permite dejar evidencia del proceso predictivo, incluso si el entrenamiento fue ejecutado desde el notebook local.

In [34]:
# Esta celda evalúa el modelo sobre el conjunto de prueba.

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import json

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1_score": float(f1_score(y_test, y_pred, zero_division=0))
}

print("Métricas del modelo:")
print(json.dumps(metrics, indent=4))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred, zero_division=0))

Métricas del modelo:
{
    "accuracy": 0.8520725407267041,
    "precision": 0.8564256584662043,
    "recall": 0.8436693255982596,
    "f1_score": 0.8499996346978586
}

Matriz de confusión:
[[60089  9752]
 [10779 58171]]

Reporte de clasificación:
              precision    recall  f1-score   support

           0       0.85      0.86      0.85     69841
           1       0.86      0.84      0.85     68950

    accuracy                           0.85    138791
   macro avg       0.85      0.85      0.85    138791
weighted avg       0.85      0.85      0.85    138791



In [35]:
# Esta celda guarda el modelo entrenado y las métricas localmente.

import os
import joblib

LOCAL_MODELS_DIR = "../models"
os.makedirs(LOCAL_MODELS_DIR, exist_ok=True)

LOCAL_MODEL_FILE = os.path.join(LOCAL_MODELS_DIR, "low_growth_logistic_model.joblib")
LOCAL_METRICS_FILE = os.path.join(LOCAL_MODELS_DIR, "low_growth_model_metrics.json")

joblib.dump(model, LOCAL_MODEL_FILE)

with open(LOCAL_METRICS_FILE, "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=4)

print("Modelo guardado localmente:", LOCAL_MODEL_FILE)
print("Métricas guardadas localmente:", LOCAL_METRICS_FILE)

Modelo guardado localmente: ../models\low_growth_logistic_model.joblib
Métricas guardadas localmente: ../models\low_growth_model_metrics.json


In [36]:
# Esta celda sube el modelo y las métricas a Amazon S3.
# Con esto, los artefactos predictivos quedan almacenados en AWS.

S3_MODEL_KEY = f"{S3_MODELS_PREFIX}low_growth_logistic_model.joblib"
S3_METRICS_KEY = f"{S3_MODELS_PREFIX}low_growth_model_metrics.json"

s3.upload_file(
    Filename=LOCAL_MODEL_FILE,
    Bucket=S3_BUCKET,
    Key=S3_MODEL_KEY
)

s3.upload_file(
    Filename=LOCAL_METRICS_FILE,
    Bucket=S3_BUCKET,
    Key=S3_METRICS_KEY
)

print("Modelo subido a:", f"s3://{S3_BUCKET}/{S3_MODEL_KEY}")
print("Métricas subidas a:", f"s3://{S3_BUCKET}/{S3_METRICS_KEY}")

Modelo subido a: s3://bigdata-salmonicultura-fabian/models/low_growth_logistic_model.joblib
Métricas subidas a: s3://bigdata-salmonicultura-fabian/models/low_growth_model_metrics.json


## 6.6 Generación y almacenamiento de predicciones

Se generan predicciones para el dataset de prueba y se almacenan en Amazon S3 en formato CSV.

Luego, estas predicciones serán registradas como una tabla externa en Athena para que puedan ser consultadas o utilizadas en un dashboard.

In [37]:
# Esta celda genera un DataFrame con las predicciones del conjunto de prueba.

df_predictions = df_test.copy()

df_predictions["predicted_low_growth_flag"] = y_pred
df_predictions["predicted_low_growth_probability"] = y_proba

prediction_columns = [
    "site",
    "unit",
    "production_week_id",
    "biomass_growth_rate",
    "low_growth_flag",
    "predicted_low_growth_flag",
    "predicted_low_growth_probability",
    "feed_per_open_biomass",
    "temperature_avg",
    "density_avg",
    "mortality_rate"
]

df_predictions_export = df_predictions[prediction_columns]

df_predictions_export.head()

,site,unit,production_week_id,biomass_growth_rate,low_growth_flag,predicted_low_growth_flag,predicted_low_growth_probability,feed_per_open_biomass,temperature_avg,density_avg,mortality_rate
0,Site_101,Unit_4997,2017,0.051875,1,1,0.669638,0.052094,13.835714,44.855715,0.000218
1,Site_101,Unit_4997,2017,-0.036161,1,1,0.739636,0.023306,13.228571,45.058606,0.000275
2,Site_101,Unit_4997,2017,4.576980,0,1,0.885976,0.017414,12.865714,15.604121,0.116150
3,Site_101,Unit_4997,2018,0.041012,1,1,0.649116,0.062395,13.507143,44.980975,0.000311
4,Site_101,Unit_4997,2018,0.100423,0,0,0.416157,0.103233,14.728571,20.482875,0.002706


In [38]:
# Esta celda guarda las predicciones localmente en formato CSV.

LOCAL_EXPORTS_DIR = "../data/processed"
os.makedirs(LOCAL_EXPORTS_DIR, exist_ok=True)

LOCAL_PREDICTIONS_FILE = os.path.join(
    LOCAL_EXPORTS_DIR,
    "ml_predictions_low_growth.csv"
)

df_predictions_export.to_csv(
    LOCAL_PREDICTIONS_FILE,
    index=False,
    encoding="utf-8"
)

print("Predicciones guardadas localmente:", LOCAL_PREDICTIONS_FILE)

Predicciones guardadas localmente: ../data/processed\ml_predictions_low_growth.csv


In [39]:
# Esta celda limpia la carpeta de predicciones en S3 y sube el archivo CSV generado.

delete_s3_prefix(S3_BUCKET, S3_ML_PREDICTIONS_PREFIX)

S3_PREDICTIONS_KEY = f"{S3_ML_PREDICTIONS_PREFIX}ml_predictions_low_growth.csv"

s3.upload_file(
    Filename=LOCAL_PREDICTIONS_FILE,
    Bucket=S3_BUCKET,
    Key=S3_PREDICTIONS_KEY
)

print("Predicciones subidas a:", f"s3://{S3_BUCKET}/{S3_PREDICTIONS_KEY}")

No había archivos para eliminar en s3://bigdata-salmonicultura-fabian/exports/ml_predictions/
Predicciones subidas a: s3://bigdata-salmonicultura-fabian/exports/ml_predictions/ml_predictions_low_growth.csv


## 6.7 Registro de predicciones como tabla Athena

Para que las predicciones puedan ser consultadas o utilizadas en un dashboard, se crea una tabla externa en Athena sobre el archivo CSV de predicciones almacenado en S3.

In [40]:
# Esta celda elimina la tabla de predicciones anterior si existe.

drop_predictions_query = f"""
DROP TABLE IF EXISTS {ATHENA_DW_DATABASE}.{ATHENA_ML_PREDICTIONS_TABLE}
"""

query_id = run_athena_query(drop_predictions_query)

print("Tabla de predicciones eliminada si existía.")
print("QueryExecutionId:", query_id)

Tabla de predicciones eliminada si existía.
QueryExecutionId: 5c2db620-38c3-4486-8800-510f09aa6a45


In [41]:
# Esta celda crea una tabla externa en Athena sobre el CSV de predicciones.
# Esta tabla puede ser usada posteriormente para consultas o dashboard.

predictions_location = f"s3://{S3_BUCKET}/{S3_ML_PREDICTIONS_PREFIX}"

create_predictions_table_query = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {ATHENA_DW_DATABASE}.{ATHENA_ML_PREDICTIONS_TABLE} (
    site string,
    unit string,
    production_week_id string,
    biomass_growth_rate double,
    low_growth_flag int,
    predicted_low_growth_flag int,
    predicted_low_growth_probability double,
    feed_per_open_biomass double,
    temperature_avg double,
    density_avg double,
    mortality_rate double
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
    'separatorChar' = ',',
    'quoteChar' = '"',
    'escapeChar' = '\\\\'
)
STORED AS TEXTFILE
LOCATION '{predictions_location}'
TBLPROPERTIES (
    'skip.header.line.count'='1'
)
"""

query_id = run_athena_query(create_predictions_table_query)

print("Tabla Athena de predicciones creada correctamente.")
print("Tabla:", f"{ATHENA_DW_DATABASE}.{ATHENA_ML_PREDICTIONS_TABLE}")
print("Ubicación:", predictions_location)
print("QueryExecutionId:", query_id)

Tabla Athena de predicciones creada correctamente.
Tabla: salmonicultura_dw_db.productive_ml_predictions
Ubicación: s3://bigdata-salmonicultura-fabian/exports/ml_predictions/
QueryExecutionId: 9905d8f1-167d-492b-93de-368815b3eed0


In [42]:
# Esta celda valida que Athena puede consultar la tabla de predicciones.

query_predictions_preview = f"""
SELECT *
FROM {ATHENA_DW_DATABASE}.{ATHENA_ML_PREDICTIONS_TABLE}
LIMIT 10
"""

query_id = run_athena_query(query_predictions_preview)

results = athena.get_query_results(QueryExecutionId=query_id)

for row in results["ResultSet"]["Rows"]:
    values = [col.get("VarCharValue", "") for col in row["Data"]]
    print(values)

['site', 'unit', 'production_week_id', 'biomass_growth_rate', 'low_growth_flag', 'predicted_low_growth_flag', 'predicted_low_growth_probability', 'feed_per_open_biomass', 'temperature_avg', 'density_avg', 'mortality_rate']
['Site_101', 'Unit_4997', '2017', '0.0518746673755983', '1', '1', '0.6696377474859277', '0.0520944346429914', '13.83571429', '44.85571544', '2.1800741225201656E-4']
['Site_101', 'Unit_4997', '2017', '-0.0361607331600913', '1', '1', '0.7396363669124757', '0.0233060352413035', '13.22857143', '45.05860584', '2.749388498075428E-4']
['Site_101', 'Unit_4997', '2017', '4.576980061237948', '0', '1', '0.8859763031496813', '0.0174137739687881', '12.86571429', '15.60412052', '0.1161498838501161']
['Site_101', 'Unit_4997', '2018', '0.0410118347809404', '1', '1', '0.6491163508761486', '0.0623953299317428', '13.50714286', '44.98097522', '3.10878350909917E-4']
['Site_101', 'Unit_4997', '2018', '0.1004231995968089', '0', '0', '0.4161568143611866', '0.103232771169293', '14.72857143',

## 6.8 Resultado de la analítica predictiva

Se implementó una etapa predictiva mixta, adaptada a las posibilidades del laboratorio AWS.

La preparación del dataset de machine learning se realizó en Athena, generando tablas persistidas en Amazon S3 para entrenamiento y prueba. Posteriormente, el modelo fue entrenado localmente con Python debido a que no se implementó Redshift ML en esta versión.

Los artefactos del modelo fueron almacenados en Amazon S3:

| Artefacto | Ubicación |
|---|---|
| Dataset ML base | `ml/dataset/` |
| Dataset de entrenamiento | `ml/train/` |
| Dataset de prueba | `ml/test/` |
| Modelo entrenado | `models/low_growth_logistic_model.joblib` |
| Métricas del modelo | `models/low_growth_model_metrics.json` |
| Predicciones | `exports/ml_predictions/` |
| Tabla Athena de predicciones | `productive_ml_predictions` |

Con esto se deja una base predictiva utilizable para consultas posteriores y para la construcción de un dashboard KPI.

# 7. Dashboard KPI – Looker Studio

La pauta original propone construir visualizaciones en Looker Studio conectadas a BigQuery. En esta implementación, el procesamiento principal fue realizado en AWS utilizando Amazon S3, Athena, Glue Data Catalog y PySpark.

Debido a que Amazon QuickSight no estaba autorizado en la cuenta del laboratorio, se utilizó Looker Studio como herramienta de visualización final. Para ello, se exportaron archivos CSV generados a partir de la capa analítica construida en AWS.

El dashboard utiliza como fuente los datos procesados y las predicciones generadas durante el pipeline. Estos archivos permiten visualizar indicadores de alimentación, biomasa, mortalidad, temperatura, densidad y riesgo de bajo crecimiento productivo.

Visualizaciones incluidas:

- alimento total por centro;
- variación de biomasa por centro;
- alimento relativo a biomasa inicial;
- mortalidad promedio;
- análisis por nivel de temperatura;
- análisis por nivel de densidad;
- predicción de bajo crecimiento productivo.

Con esto se completa la etapa de visualización del pipeline, utilizando Looker Studio como alternativa práctica para presentar los KPIs finales del proyecto.

## 7.1 Preparación de archivos para Looker Studio

Para construir el dashboard se utilizarán archivos CSV exportados desde la capa analítica generada en AWS.

Se utilizarán dos archivos principales:

| Archivo | Descripción | Uso |
|---|---|---|
| `productive_kpi_processed.csv` | Dataset procesado principal | KPIs de alimentación, biomasa, mortalidad, temperatura y densidad |
| `ml_predictions_low_growth.csv` | Predicciones del modelo | Visualización del riesgo de bajo crecimiento productivo |

Los archivos serán guardados localmente en la carpeta:

```text
dashboard/
```

Estructura esperada:

```text
bigdata-salmonicultura-aws/
│
├── dashboard/
│   ├── productive_kpi_processed.csv
│   └── ml_predictions_low_growth.csv
```

In [58]:
# Esta celda crea la carpeta local dashboard/.
# En esta carpeta se guardarán los archivos CSV que serán usados como fuente en Looker Studio.

import os

LOCAL_DASHBOARD_DIR = "../dashboard"

os.makedirs(LOCAL_DASHBOARD_DIR, exist_ok=True)

print("Carpeta dashboard preparada en:")
print(LOCAL_DASHBOARD_DIR)

Carpeta dashboard preparada en:
../dashboard


## 7.2 Exportación de KPIs por centro

En lugar de exportar el dataset procesado completo, se generará un archivo resumido por centro.

Esta decisión permite reducir el tamaño del archivo y facilitar la carga en Looker Studio, que posee un límite de tamaño para archivos CSV cargados manualmente.

El archivo generado será:

```text
dashboard_site_kpis.csv
```

Este resumen será la fuente principal para las visualizaciones productivas del dashboard.

In [59]:
# Esta consulta genera un dataset resumido por centro para Looker Studio.
# Es más liviano que exportar toda la tabla procesada completa y contiene los KPIs principales.

query_dashboard_site_kpis = f"""
SELECT
    site,
    CAST(ROUND(SUM(feed_weight_numeric), 2) AS DECIMAL(18,2)) AS total_feed_weight,
    CAST(ROUND(SUM(biomass_variation), 2) AS DECIMAL(18,2)) AS total_biomass_variation,
    CAST(ROUND(AVG(feed_per_open_biomass), 6) AS DECIMAL(18,6)) AS avg_feed_per_open_biomass,
    CAST(ROUND(AVG(biomass_growth_rate), 6) AS DECIMAL(18,6)) AS avg_biomass_growth_rate,
    CAST(ROUND(AVG(mortality_rate) * 100, 4) AS DECIMAL(18,4)) AS avg_mortality_percent,
    CAST(ROUND(AVG(temperature_avg), 2) AS DECIMAL(18,2)) AS avg_temperature,
    CAST(ROUND(AVG(density_avg), 2) AS DECIMAL(18,2)) AS avg_density,
    COUNT(*) AS total_records
FROM {ATHENA_DW_DATABASE}.{ATHENA_PROCESSED_TABLE}
WHERE feed_weight_numeric IS NOT NULL
GROUP BY site
ORDER BY total_feed_weight DESC
"""

df_dashboard_site_kpis = athena_query_to_dataframe(query_dashboard_site_kpis)

print("Registros:", df_dashboard_site_kpis.shape[0])
print("Columnas:", df_dashboard_site_kpis.shape[1])

df_dashboard_site_kpis.head()

Registros: 99
Columnas: 9


,site,total_feed_weight,total_biomass_variation,avg_feed_per_open_biomass,avg_biomass_growth_rate,avg_mortality_percent,avg_temperature,avg_density,total_records
0,Site_038,92833689.00,56913023.42,0.069052,0.058140,0.1478,11.34,7.52,17086
1,Site_059,85706240.32,64638315.85,0.068641,0.065641,0.1616,11.17,7.69,12021
2,Site_012,62291237.00,45380889.96,0.067100,0.062104,0.1812,11.10,6.89,10399
3,Site_027,50881250.61,36194231.73,0.067642,0.057494,0.3049,12.03,6.10,9927
4,Site_049,48557404.59,33210684.70,0.066339,0.057903,0.1559,11.88,6.60,9529


## 7.3 Guardado del resumen por centro

El resultado anterior se guarda como archivo CSV dentro de la carpeta local `dashboard/`.

Este archivo será usado en Looker Studio para construir gráficos por centro, como alimento total, biomasa ganada, mortalidad promedio y alimento relativo a biomasa.

In [60]:
# Esta celda guarda el resumen por centro en formato CSV para usarlo en Looker Studio.

import os

LOCAL_DASHBOARD_DIR = "../dashboard"
os.makedirs(LOCAL_DASHBOARD_DIR, exist_ok=True)

LOCAL_SITE_KPIS_FILE = os.path.join(
    LOCAL_DASHBOARD_DIR,
    "dashboard_site_kpis.csv"
)

df_dashboard_site_kpis.to_csv(
    LOCAL_SITE_KPIS_FILE,
    index=False,
    encoding="utf-8"
)

print("Archivo guardado:")
print(LOCAL_SITE_KPIS_FILE)

Archivo guardado:
../dashboard\dashboard_site_kpis.csv


## 7.4 Exportación de KPIs por nivel de temperatura

Se genera un segundo archivo resumido, agrupado por nivel de temperatura.

Este archivo permitirá construir gráficos que relacionen la alimentación, el crecimiento de biomasa y la mortalidad con las condiciones de temperatura.

El archivo generado será:

```text
dashboard_temperature_kpis.csv
```

In [61]:
# Esta consulta genera un resumen por nivel de temperatura para el dashboard.

query_dashboard_temperature_kpis = f"""
SELECT
    temperature_level,
    COUNT(*) AS total_records,
    CAST(ROUND(SUM(feed_weight_numeric), 2) AS DECIMAL(18,2)) AS total_feed_weight,
    CAST(ROUND(AVG(feed_per_open_biomass), 6) AS DECIMAL(18,6)) AS avg_feed_per_open_biomass,
    CAST(ROUND(AVG(biomass_growth_rate), 6) AS DECIMAL(18,6)) AS avg_biomass_growth_rate,
    CAST(ROUND(AVG(mortality_rate) * 100, 4) AS DECIMAL(18,4)) AS avg_mortality_percent
FROM {ATHENA_DW_DATABASE}.{ATHENA_PROCESSED_TABLE}
WHERE temperature_level IS NOT NULL
  AND feed_weight_numeric IS NOT NULL
GROUP BY temperature_level
ORDER BY temperature_level
"""

df_dashboard_temperature_kpis = athena_query_to_dataframe(query_dashboard_temperature_kpis)

LOCAL_TEMPERATURE_KPIS_FILE = os.path.join(
    LOCAL_DASHBOARD_DIR,
    "dashboard_temperature_kpis.csv"
)

df_dashboard_temperature_kpis.to_csv(
    LOCAL_TEMPERATURE_KPIS_FILE,
    index=False,
    encoding="utf-8"
)

print("Archivo guardado:")
print(LOCAL_TEMPERATURE_KPIS_FILE)

df_dashboard_temperature_kpis.head()

Archivo guardado:
../dashboard\dashboard_temperature_kpis.csv


,temperature_level,total_records,total_feed_weight,avg_feed_per_open_biomass,avg_biomass_growth_rate,avg_mortality_percent
0,alta,132892,104159228.89,0.235468,0.199869,1.0271
1,baja,126067,171335096.03,667037893.338962,20436196123.715508,0.6104
2,media,376972,850354836.10,459563325.721665,13497803061.235998,1.0192
3,sin_dato,99,7239.20,0.025714,-0.998377,2.7386


## 7.5 Exportación de KPIs por nivel de densidad

Se genera un tercer archivo resumido, agrupado por nivel de densidad.

Este archivo permitirá analizar alimentación, crecimiento de biomasa y mortalidad según la densidad productiva de las unidades de cultivo.

El archivo generado será:

```text
dashboard_density_kpis.csv
```

In [62]:
# Esta consulta genera un resumen por nivel de densidad para el dashboard.

query_dashboard_density_kpis = f"""
SELECT
    density_level,
    COUNT(*) AS total_records,
    CAST(ROUND(SUM(feed_weight_numeric), 2) AS DECIMAL(18,2)) AS total_feed_weight,
    CAST(ROUND(AVG(feed_per_open_biomass), 6) AS DECIMAL(18,6)) AS avg_feed_per_open_biomass,
    CAST(ROUND(AVG(biomass_growth_rate), 6) AS DECIMAL(18,6)) AS avg_biomass_growth_rate,
    CAST(ROUND(AVG(mortality_rate) * 100, 4) AS DECIMAL(18,4)) AS avg_mortality_percent
FROM {ATHENA_DW_DATABASE}.{ATHENA_PROCESSED_TABLE}
WHERE density_level IS NOT NULL
  AND feed_weight_numeric IS NOT NULL
GROUP BY density_level
ORDER BY density_level
"""

df_dashboard_density_kpis = athena_query_to_dataframe(query_dashboard_density_kpis)

LOCAL_DENSITY_KPIS_FILE = os.path.join(
    LOCAL_DASHBOARD_DIR,
    "dashboard_density_kpis.csv"
)

df_dashboard_density_kpis.to_csv(
    LOCAL_DENSITY_KPIS_FILE,
    index=False,
    encoding="utf-8"
)

print("Archivo guardado:")
print(LOCAL_DENSITY_KPIS_FILE)

df_dashboard_density_kpis.head()

Archivo guardado:
../dashboard\dashboard_density_kpis.csv


,density_level,total_records,total_feed_weight,avg_feed_per_open_biomass,avg_biomass_growth_rate,avg_mortality_percent
0,alta,112049,40175250.61,0.125522,0.057702,1.0456
1,baja,413568,683174136.09,627666730.613903,18692934609.214110,0.8876
2,media,110413,402507013.52,0.110920,0.100612,1.0739


## 7.6 Descarga del archivo de predicciones

El archivo de predicciones fue generado durante la etapa de analítica predictiva y almacenado en Amazon S3.

Se descargará una copia local para cargarla en Looker Studio como fuente del dashboard predictivo.

El archivo esperado es:

```text
ml_predictions_low_growth.csv
```

In [63]:
# Esta celda descarga desde S3 el archivo de predicciones generado en la etapa de ML.
# El archivo será usado como fuente de datos para el dashboard predictivo en Looker Studio.

LOCAL_ML_PREDICTIONS_FILE = os.path.join(
    LOCAL_DASHBOARD_DIR,
    "ml_predictions_low_growth.csv"
)

S3_PREDICTIONS_KEY = "exports/ml_predictions/ml_predictions_low_growth.csv"

s3.download_file(
    Bucket=S3_BUCKET,
    Key=S3_PREDICTIONS_KEY,
    Filename=LOCAL_ML_PREDICTIONS_FILE
)

print("CSV de predicciones descargado en:")
print(LOCAL_ML_PREDICTIONS_FILE)

CSV de predicciones descargado en:
../dashboard\ml_predictions_low_growth.csv


## 7.7 Validación de archivos locales

Antes de cargar los datos en Looker Studio, se valida que todos los archivos CSV necesarios existan en la carpeta local `dashboard/`.

Archivos esperados:

- `dashboard_site_kpis.csv`
- `dashboard_temperature_kpis.csv`
- `dashboard_density_kpis.csv`
- `ml_predictions_low_growth.csv`

In [64]:
# Esta celda valida que los archivos CSV necesarios para Looker Studio existen localmente.

dashboard_files = [
    LOCAL_SITE_KPIS_FILE,
    LOCAL_TEMPERATURE_KPIS_FILE,
    LOCAL_DENSITY_KPIS_FILE,
    LOCAL_ML_PREDICTIONS_FILE
]

for file_path in dashboard_files:
    if os.path.exists(file_path):
        size_kb = os.path.getsize(file_path) / 1024
        print(f"Archivo encontrado: {file_path} | Tamaño: {size_kb:.2f} KB")
    else:
        print("Archivo faltante:", file_path)

Archivo encontrado: ../dashboard\dashboard_site_kpis.csv | Tamaño: 7.10 KB
Archivo encontrado: ../dashboard\dashboard_temperature_kpis.csv | Tamaño: 0.35 KB
Archivo encontrado: ../dashboard\dashboard_density_kpis.csv | Tamaño: 0.28 KB
Archivo encontrado: ../dashboard\ml_predictions_low_growth.csv | Tamaño: 17311.49 KB


## 7.8 Carga de datos en Looker Studio

A partir de este punto, el trabajo se realiza directamente en Looker Studio.

Se deben crear fuentes de datos mediante carga de archivo CSV.

Archivos a cargar:

```text
dashboard_site_kpis.csv
dashboard_temperature_kpis.csv
dashboard_density_kpis.csv
ml_predictions_low_growth.csv
```

Pasos generales:

```text
Looker Studio
↓
Crear
↓
Fuente de datos
↓
Subida de archivo
↓
Crear conjunto de datos
↓
Añadir archivo CSV
↓
Conectar
↓
Añadir al informe
```

Se recomienda comenzar con `dashboard_site_kpis.csv`, ya que será la fuente principal del dashboard productivo.

## 7.9 Revisión de tipos de datos en Looker Studio

Después de cargar los CSV, se deben revisar los tipos de datos detectados por Looker Studio.

Para `dashboard_site_kpis.csv`, los campos numéricos principales son:

```text
total_feed_weight
total_biomass_variation
avg_feed_per_open_biomass
avg_biomass_growth_rate
avg_mortality_percent
avg_temperature
avg_density
total_records
```

El campo `site` debe quedar como dimensión.

Para `dashboard_temperature_kpis.csv`, el campo `temperature_level` debe quedar como dimensión y las métricas como valores numéricos.

Para `dashboard_density_kpis.csv`, el campo `density_level` debe quedar como dimensión y las métricas como valores numéricos.

Para `ml_predictions_low_growth.csv`, los campos numéricos principales son:

```text
biomass_growth_rate
low_growth_flag
predicted_low_growth_flag
predicted_low_growth_probability
feed_per_open_biomass
temperature_avg
density_avg
mortality_rate
```

Los campos `site`, `unit` y `production_week_id` deben quedar como dimensiones.

## 7.10 Dashboard productivo

La primera página del dashboard utilizará principalmente la fuente:

```text
dashboard_site_kpis.csv
```

Visualizaciones recomendadas:

| Visualización | Dimensión | Métrica |
|---|---|---|
| Tarjeta: alimento total | Sin dimensión | `SUM(total_feed_weight)` |
| Tarjeta: biomasa total ganada | Sin dimensión | `SUM(total_biomass_variation)` |
| Tarjeta: mortalidad promedio | Sin dimensión | `AVG(avg_mortality_percent)` |
| Barras: alimento total por centro | `site` | `total_feed_weight` |
| Barras: variación de biomasa por centro | `site` | `total_biomass_variation` |
| Barras: alimento relativo por centro | `site` | `avg_feed_per_open_biomass` |
| Barras: mortalidad promedio por centro | `site` | `avg_mortality_percent` |

También se pueden agregar gráficos complementarios con:

```text
dashboard_temperature_kpis.csv
dashboard_density_kpis.csv
```

Visualizaciones complementarias:

| Archivo | Dimensión | Métrica |
|---|---|---|
| `dashboard_temperature_kpis.csv` | `temperature_level` | `avg_feed_per_open_biomass` |
| `dashboard_temperature_kpis.csv` | `temperature_level` | `avg_biomass_growth_rate` |
| `dashboard_density_kpis.csv` | `density_level` | `avg_feed_per_open_biomass` |
| `dashboard_density_kpis.csv` | `density_level` | `avg_mortality_percent` |

[Ver dashboard KPI en Looker Studio](https://datastudio.google.com/s/s_DCOggeXLU)

## 7.11 Dashboard predictivo

La segunda página del dashboard utilizará la fuente:

```text
ml_predictions_low_growth.csv
```

Visualizaciones recomendadas:

| Visualización | Dimensión | Métrica |
|---|---|---|
| Tarjeta: probabilidad promedio de bajo crecimiento | Sin dimensión | `AVG(predicted_low_growth_probability)` |
| Barras: riesgo promedio por centro | `site` | `AVG(predicted_low_growth_probability)` |
| Tabla: unidades con mayor riesgo | `site`, `unit`, `production_week_id` | `predicted_low_growth_probability` |
| Barras: crecimiento real por centro | `site` | `AVG(biomass_growth_rate)` |
| Barras: bajo crecimiento predicho | `predicted_low_growth_flag` | `COUNT(predicted_low_growth_flag)` |

La tabla de unidades con mayor riesgo debe ordenarse de mayor a menor por:

```text
predicted_low_growth_probability
```

## 8. Resumen general del flujo implementado y limitaciones del trabajo

A lo largo de los notebooks desarrollados se implementó un pipeline batch completo, adaptando la pauta original basada en Google Cloud Platform a una arquitectura equivalente en AWS. El objetivo fue construir un flujo funcional que permitiera cargar datos productivos históricos, almacenarlos en la nube, crear una capa RAW, transformar los datos, generar indicadores, construir una capa analítica, preparar una etapa predictiva y finalmente visualizar los resultados en un dashboard.

### 8.1 Resumen de etapas desarrolladas

El trabajo fue dividido en varios notebooks para mantener ordenado el desarrollo del pipeline.

| Notebook | Etapas trabajadas | Descripción |
|---|---|---|
| `pipeline_batch_salmonicultura01.ipynb` | Caso de negocio, ingesta y tabla RAW | Se definió el problema de negocio, se cargó el dataset en Amazon S3 y se creó una tabla RAW en Athena / Glue Data Catalog. |
| `pipeline_batch_salmonicultura02.ipynb` | Preparación y transformación | Se utilizó PySpark para leer los datos desde S3, limpiar columnas, convertir tipos, validar datos y generar indicadores productivos con foco en alimentación. |
| `pipeline_batch_salmonicultura03.ipynb` | Data Warehouse, persistencia, analítica y ML | Se creó una capa analítica en Athena sobre los datos procesados, se ejecutaron consultas KPI, se prepararon datasets de entrenamiento y prueba, y se generó una etapa predictiva básica. |
| Looker Studio | Dashboard KPI | Se construyó un dashboard final utilizando archivos CSV exportados desde la capa analítica generada en AWS. |

### 8.2 Componentes ejecutados en AWS

La mayor parte del pipeline se mantuvo dentro de servicios AWS, respetando la lógica de una arquitectura cloud.

| Componente | Servicio AWS utilizado | Rol dentro del pipeline |
|---|---|---|
| Almacenamiento RAW | Amazon S3 | Guardar el archivo original sin modificaciones. |
| Tabla RAW | Amazon Athena / Glue Data Catalog | Consultar el dataset original desde S3. |
| Transformación batch | PySpark conectado a S3 | Leer datos desde S3, transformar y generar indicadores. |
| Capa procesada | Amazon S3 | Guardar el dataset procesado en formato Parquet. |
| Data Warehouse analítico | Athena / Glue Data Catalog | Crear una tabla consultable sobre la capa procesada. |
| Consultas KPI | Athena SQL | Calcular indicadores de alimentación, biomasa, mortalidad, temperatura y densidad. |
| Persistencia ML | Amazon S3 | Guardar datasets de entrenamiento, prueba, predicciones, métricas y modelo. |

De esta forma, AWS funcionó como la base principal del flujo: almacenamiento, consulta, procesamiento batch, persistencia y capa analítica.

### 8.3 Componentes ejecutados localmente

Algunas partes debieron realizarse localmente por restricciones del laboratorio y de permisos disponibles.

| Componente local | Motivo |
|---|---|
| Notebook de control | Se utilizó como herramienta de ejecución, documentación y evidencia del pipeline. |
| PySpark local | Permitió ejecutar la transformación batch leyendo y escribiendo datos en S3. |
| Entrenamiento del modelo predictivo | Se realizó localmente porque no se implementó Redshift ML ni SageMaker por limitaciones de permisos, complejidad y consumo de créditos. |
| Looker Studio con CSV | Se utilizó como alternativa final de visualización, ya que QuickSight no estaba autorizado y Redshift fue descartado por costo. |

Aunque estas partes se ejecutaron localmente, los datos de entrada y salida principales quedaron vinculados a AWS, manteniendo la trazabilidad del pipeline.

### 8.4 Limitaciones del dataset utilizado

Una limitación importante del trabajo es que el archivo utilizado no corresponde a un caso Big Data en sentido estricto por volumen. El dataset productivo histórico permitió aplicar una metodología cercana a Big Data, pero no representa por sí solo el volumen, velocidad y variedad esperados en un escenario real con video submarino, sensores, eventos de alimentación automatizada y múltiples centros generando datos continuamente.

El proyecto original planteaba una solución orientada a alimentación automatizada y monitoreo submarino. Sin embargo, para esta entrega se trabajó con la data disponible, principalmente registros productivos históricos semanales. Esto permitió construir una primera aproximación batch, pero dejó fuera elementos relevantes como:

- eventos detallados de alimentación en tiempo real;
- registros horarios de sistemas automáticos;
- comportamiento de peces observado por video;
- integración con sensores operacionales;
- procesamiento streaming;
- análisis avanzado de imágenes o video.

Por lo tanto, el resultado debe entenderse como una primera implementación acotada del flujo, no como una solución productiva completa.

### 8.5 Limitaciones del análisis

Los indicadores generados permitieron analizar variables como alimento entregado, biomasa, mortalidad, temperatura y densidad. Sin embargo, el análisis tiene limitaciones porque las variables disponibles no permiten explicar completamente la eficiencia alimentaria ni el comportamiento productivo.

Por ejemplo, `Feed Weight` permite aproximarse al alimento entregado, pero no entrega el mismo nivel de detalle que un sistema de alimentación automatizada real. Tampoco se cuenta con información sobre horarios de alimentación, apetito observado, pellet no consumido, comportamiento de los peces, oxígeno, corrientes, tratamientos sanitarios u otros factores relevantes.

Por ello, los KPIs generados deben interpretarse como indicadores exploratorios. Son útiles para probar el flujo analítico, comparar centros y detectar patrones generales, pero no bastan para tomar decisiones productivas precisas sin una revisión técnica más profunda y datos complementarios.

### 8.6 Limitaciones del modelo predictivo

La etapa predictiva también debe considerarse una prueba metodológica más que un modelo final de negocio. El modelo fue construido para clasificar semanas con bajo crecimiento de biomasa, usando variables derivadas como alimentación relativa, temperatura, densidad, mortalidad y biomasa inicial.

Sin embargo, los resultados no fueron completamente satisfactorios y requieren una nueva revisión. Esto puede deberse a varios factores:

- la variable objetivo fue creada de forma derivada y simplificada;
- el dataset no contiene todas las variables que explican realmente el crecimiento;
- la relación entre alimentación y crecimiento puede tener desfases temporales;
- la mortalidad, densidad o temperatura pueden no capturar toda la complejidad productiva;
- el modelo utilizado fue básico y no optimizado;
- el objetivo predictivo puede requerir una definición más robusta desde el punto de vista del negocio.

Aun así, la etapa sirvió para implementar el flujo completo de analítica predictiva: preparación del dataset, separación de entrenamiento y prueba, entrenamiento del modelo, evaluación, generación de predicciones y almacenamiento de resultados en Amazon S3.

### 8.7 Evaluación general del trabajo

A pesar de las limitaciones, el proyecto permitió implementar un flujo completo inspirado en una arquitectura Big Data batch. Se logró demostrar la lógica de trabajo desde la ingesta hasta la visualización:

```text
Dataset histórico
        ↓
Amazon S3 RAW
        ↓
Athena / Glue Data Catalog RAW
        ↓
PySpark
        ↓
S3 processed
        ↓
Athena Data Warehouse
        ↓
Consultas KPI
        ↓
Datasets ML y predicciones
        ↓
Looker Studio

El trabajo no representa todavía una solución Big Data productiva completa, pero sí permite validar la metodología, probar herramientas cloud, organizar los datos por capas y construir una base escalable para futuras mejoras.

En una siguiente versión, el proyecto podría fortalecerse incorporando datos reales de alimentación automatizada, mayor volumen histórico, sensores ambientales, video submarino, procesamiento streaming y modelos predictivos más robustos entrenados directamente en servicios cloud especializados.

En conclusión, el pipeline desarrollado cumple el objetivo académico de implementar un flujo batch equivalente al solicitado en la pauta, adaptado a AWS y a los datos disponibles. Además, deja una base funcional para extender el proyecto hacia un escenario Big Data más completo y cercano al planteamiento original.